In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import solve, schur
import scipy as sp
from scipy.integrate import solve_ivp
import sys, os, pickle
from joblib import Parallel, delayed, cpu_count
from matplotlib.animation import FuncAnimation
from utility import orbit, call_method
from models import Mckean_Vlasov
import datetime

In [ ]:
if __name__ == "__main__":
    param_file = "./config_models/mckean_vlasov_param_1.in"  # JSON file containing model parameters
    model = Mckean_Vlasov(param_file)
    print("Loaded parameters:", model.n_z)



    f = model.dydt_per
    J = model.jacobian_per

    z, z_centers, h = model.mesh1D  # Get the mesh and centers
    sigma = 1

    # model.V = model.V_kuramoto
    model.alpha_shift = 0*np.pi
    model.I = 1.0
    # M = 1/(sigma*np.sqrt(2*np.pi))
    #start from a gaussian
    # y0 = M*np.exp(-z_centers**2/2*sigma**2)
    y0 = np.sin(2*np.pi*z_centers/(model.xmax - model.xmin))+1
    model.m0 = float(h*np.ones_like(y0) @ y0)

    # sol0 = solve_ivp(f, (0, 10*model.T_ini), y0, method='RK45',
    #              rtol=1e-7, atol=1e-9,
    #              t_eval=np.linspace(0, 10*model.T_ini, 100))
    
    sol = solve_ivp(f, (0, 100*model.T_ini), y0, method='BDF', jac = J, 
                 rtol=1e-4, atol=1e-6,
                 t_eval=np.linspace(0, 100*model.T_ini, 500))

In [ ]:
def Bernoulli(z):
    "The Bernoulli function"
    mask = np.abs(z)<=1e-5
    B = np.zeros_like(z)
    print(z[mask])
    print(z[~mask])
    B[mask] = 1-z[mask]/2+z[mask]*z[mask]/12-(z[mask]**4)/720
    B[~mask] = z[~mask]/(np.exp(z[~mask])-1)
    print('B not mask = ', B[~mask])
    
    print(z[~mask]/(np.exp(z[~mask])-1))

    # return np.where(np.abs(z)<=1e-5,1-z/2+z*z/12-(z**4)/720, z/(np.exp(z)-1))
    return B

tab = np.array([0,1])
Bernoulli(tab)

In [ ]:
plt.figure()
fig, ax = plt.subplots()
line, = ax.plot(z_centers, sol.y[:, 0])
ax.set_xlim(z_centers[0], z_centers[-1])
ax.set_ylim(0, np.max(sol.y)*1.1)
ax.set_title('Solution Evolution')
ax.set_xlabel('z')
ax.set_ylabel('y(z,t)')
def update(frame):
    line.set_ydata(sol.y[:, frame])
    ax.set_title(f'Solution Evolution at t={sol.t[frame]:.2f}')
    return line,
ani = FuncAnimation(fig, update, frames=len(sol.t), blit=True, interval=2)
ani.save('solution_with_kuramoto_potential.gif', writer='imagemagick')
plt.show(),


In [ ]:
def run(model,f, J,n_z,orbit_method,p0,T, y0, filename=None, t_scale=True):
    epsilon = model.precision
    model.n_z = n_z
    model.p0 = p0 #Size of the dominant subspace
    model.update_params(**{'n_z': n_z}) #Update the model parameters
    #Initialization
    z, z_centers, h = model.mesh1D
    
    T_unit = 1.0
    t_span = (0, 6*T)
    H = h*np.ones_like(y0)
    model.m0 = H @ y0
    print('Mass at initial point:', model.m0)
    #We integrate sufficiently the equation to find a good starting point
    phi_t = solve_ivp(f, t_span, y0, method='BDF', jac = J,
                     rtol=1e-7, atol=1e-9,
                     t_eval= [6*T])#np.linspace(0, 10, 100))
    
    y_T = phi_t.y[:,-1] #Using phi(y0,T0) as a starting point
    
  
    print('Mass at the starting point:', H@y_T)
    orbit_finder = orbit(f,y0,T, J ,2, solve_ivp, model.method, 10000,model.max_iter, epsilon)
    
    V_0 = np.eye(len(y0))[:,:p0+model.pe]#Initial guess of the subspace
    #The arguments to pass to the orbit_finder method
    args_func = {
    "y0": y_T,
    "T_0": T,
    "model": model,
    "f_unscaled": f,
    "jac_unscaled": J,
    "alpha_0": model.alpha,#0.001*np.ones_like(y_T),
    "Max_iter": model.max_iter,
    "epsilon": epsilon,
    "subsp_iter": model.subsp_iter,
    "l": model.picard_iter,
    "Ve_0": V_0,
    "p0": p0,
    "pe": model.pe,
    "rho": model.rho,
    "phase_cond": 2,
    "l": model.picard_iter,
    "full_sub_iter": model.full_sub_iter, # Use the full subspace iteration if True for the subspace iteration with projection
    "h": h
    }
    method_to_call= getattr(orbit_finder, orbit_method)

    return call_method(method_to_call, **args_func)

In [ ]:
nz = model.n_z
model.alpha = 0.0
# model.p0 = 10
y0 = np.ones(nz-1)
T = model.T_ini
model.precision = 1e-12
k, T_by_iter, y_by_iter, Norm_B, Abs_Err, Rel_Err, converged, mass = run(model,f,J,nz,"Newton_mass_conserv4",
                                                                         model.p0,T,y0, filename=None, t_scale=True);

#save the results to a pickle file
# with open('./Results/mckean_vlasov_param_1.in/Np_spence_18_03_2026.pkl', 'wb') as file:
#     data = {
#         'k': k,
#         'T_by_iter': T_by_iter,
#         'y_by_iter': y_by_iter,
#         'Norm_B': Norm_B,
#         'Abs_Err': Abs_Err,
#         'Rel_Err': Rel_Err,
#         'converged': converged,                
#         'Delta_mass': mass,
#         'p0': model.p0,
#         'alpha': model.alpha,
#         'n_z': model.n_z
#     }
#     pickle.dump(data, file)

In [ ]:
rho =  np.random.rand(model.n_z-1)
Jac = model.jacobian_per(0, rho)
f = model.dydt_per
#Finite difference approximation of the Jacobian
def compute_jacobian(f, x, epsi=1e-5):
    """    Compute the Jacobian of a vector function f at point x using finite differences.
    """

    #Do not use explilicite loop to compute the Jacobian
    n = len(x)
    m = len(f(0,x))
    J = np.zeros((m, n))

    for i in range(m):
        for j in range(n):
            x_plus = np.copy(x)
            x_minus = np.copy(x)
            x_plus[j] += epsi
            x_minus[j] -= epsi
            J[i, j] = (f(0,x_plus)[i] - f(0,x_minus)[i]) / (2 * epsi)
      
    return J

# J_num = compute_jacobian(lambda r: f(0, r), rho, h=1e-8)
J_num = compute_jacobian(f,rho, epsi=1e-5)
#Compare the two Jacobians
print('Difference between analytical and numerical Jacobian:', np.linalg.norm(Jac - J_num))
#Check that the Jacobian is correct using the definition

# model.jacobian(0, rho) @ (rho) - f(0, rho)


In [ ]:
mask = np.abs(Jac) > 1e-10
plt.spy(Jac*mask, markersize=10)
plt.show()

# plt.spy(J_nonlin, markersize=10)
mask = np.abs(J_num) > 1e-10
plt.spy(J_num*mask, markersize=10)


In [ ]:
#Checking the jacobian function
R = []
H = [1e-1,1e-2, 1e-3,1e-4,1e-5,1e-6,1e-7,1e-8]
for eps in H:
    r = f(0, rho + eps*rho) - f(0, rho)  - Jac @ (eps*rho)
    R.append(np.linalg.norm(r))
print("Residuals for different epsilons:", R)

plt.figure(figsize=(10, 6))
plt.plot(H, R, marker='o')
#plot the slope h***2
plt.plot(H, [R[0] * (h / H[0])**2 for h in H], marker='*',linestyle='--', color='red', label='Slope ~ h^2')
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Epsilon (h)')
plt.ylabel('Residual Norm')
plt.legend()
plt.title('Residual Norm vs Epsilon')
plt.grid(True)